# Experiment 4 · Cross-dataset LoRA transfer (foundation models)

This experiment measures how well a LoRA-adapted foundation model **transfers across datasets**. It is **evaluation-only**: no training happens here. For each foundation model we take a LoRA checkpoint that was fine-tuned on a *source* dataset in **Experiment 3** (few-shot LoRA, §5.3) and evaluate it, unchanged, on the held-out **test split of a different *target* dataset**.

Sweeping every ordered (source, target) pair over the four datasets — DDTI, TN3K, ThyroidXL, Stanford AIMI — fills the **off-diagonal** cells of a 4×4 transfer matrix (the on-diagonal in-domain cells come from Experiment 3). Each cell is repeated at two training fractions (`f = 0.25`, `0.5`), so the source LoRA was learned from that fraction of the source-dataset training set. Every image is prompted with its ground-truth bounding box, and by default the source-dataset normalization statistics are used at inference (`--norm_stats src`). The resulting matrices are the numbers behind the cross-dataset LoRA table (tab:exp45) in the paper.

In [ ]:
# Move to the repository root (the directory that contains pyproject.toml) so that
# the `thyroidbench` package is importable and the relative --data_root / --src_ckpt_root
# defaults resolve correctly.
import os
from pathlib import Path

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists():
        os.chdir(candidate)
        break
print("Repository root:", Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

This experiment loads checkpoints and weights produced elsewhere; it trains nothing.

1. **Experiment 3 LoRA checkpoints.** Each cross-eval run loads the source-dataset LoRA checkpoint written by Experiment 3 (few-shot LoRA). They are resolved from `--src_ckpt_root` (default `experiments/exp3_fewshot_lora/results`) as `fewshot_<model>_<src_dataset>_f<fraction>/fewshot_<model>_<src_dataset>_f<fraction>_best.pt`. Run Experiment 3 first (or point `--src_ckpt_root` at wherever those checkpoints live) so that every source/model/fraction checkpoint exists.
2. **Foundation-model weights.** The base foundation weights are **not** shipped with the repository. Download the published checkpoints and place them where each wrapper in `thyroidbench/models/` expects them (see the Experiment 2 notebook for the full table). The LoRA adapter is applied on top of these base weights.
3. **Preprocessed datasets** under `--data_root` (default `data`), with patient-level splits, exactly as used by Experiments 2 and 3.

## Run

In [ ]:
datasets = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
for model in ['sam2', 'medsam', 'medsam2', 'sam3']:
    for src in datasets:
        for tgt in datasets:
            if src == tgt: continue
            for frac in [0.25, 0.5]:
                !python experiments/exp4_crossdataset/foundation/run_crosseval.py --model {model} --src_dataset {src} --tgt_dataset {tgt} --fraction {frac}

## Results

The off-diagonal cross-eval cells are collated with the on-diagonal in-domain references (pulled from Experiment 3) into a single master CSV by `aggregate_exp4_5.py`. The cell below loads it and pivots mean DSC into the 4×4 source→target transfer matrix per (model, fraction) — the numbers behind tab:exp45. Run `python experiments/exp4_crossdataset/foundation/compute_stats.py` to regenerate the bootstrap CIs and pairwise Wilcoxon tables under `results/stats/`.

In [ ]:
import pandas as pd

master = pd.read_csv('experiments/exp4_crossdataset/foundation/results/exp4_5_master.csv')
display(master.head(20))

# Mean DSC source -> target transfer matrix, per (model, fraction).
for (model, frac), grp in master.groupby(['model', 'fraction']):
    mat = grp.pivot_table(index='src_dataset', columns='tgt_dataset', values='dice_mean')
    print(f'\nmodel={model}  fraction={frac}  (rows=source, cols=target)')
    display(mat.round(4))